# 11 · Graphs

> Nodi + archi. Negli esercizi il grafo è quasi sempre una **griglia** (celle vicine) o una **lista di archi** da trasformare in lista di adiacenza. Gli strumenti: DFS, BFS (anche multi-sorgente), ordinamento topologico, Union-Find.
> Idee per i singoli problemi: [Alberi e grafi › Grafi e griglie](../Fundamentals/Alberi%20e%20grafi.md#grafi-e-griglie) · [Ordinamento topologico](../Fundamentals/Alberi%20e%20grafi.md#ordinamento-topologico) · [Union-Find](../Fundamentals/Big-O%20e%20strutture%20dati.md#union-find-disjoint-set)

## Problemi → cosa ti serve

| Problema | Concetti | Sintassi |
|---|---|---|
| Number of Islands | DFS/BFS da ogni cella di terra non visitata; conta gli avvii | direzioni, controllo bordi, `visited` o marcatura in-place |
| Max Area of Island | la DFS restituisce la dimensione | `1 + sum(dfs(...) for ...)` |
| Clone Graph | mappa originale → copia | `dict`, DFS o BFS |
| Walls and Gates | BFS multi-sorgente da tutte le porte | `deque` iniziale con più celle |
| Rotting Oranges | BFS multi-sorgente per livelli = minuti | `for _ in range(len(queue))`, contatore delle fresche |
| Pacific Atlantic Water Flow | ragiona al contrario: parti dai bordi e "sali" | due set `visited`, intersezione `a & b` |
| Surrounded Regions | marca le `O` collegate al bordo, poi converti le altre | DFS dai bordi, marcatore temporaneo |
| Course Schedule | ciclo in un grafo diretto | DFS a 3 stati o Kahn (gradi entranti) |
| Course Schedule II | ordinamento topologico | Kahn: `deque` dei nodi con grado 0 |
| Graph Valid Tree | `n − 1` archi + connesso + senza cicli | Union-Find o DFS con `parent` |
| Number of Connected Components | conta le unioni riuscite (o gli avvii di DFS) | Union-Find |
| Redundant Connection | il primo arco che unisce due nodi già connessi | `find(a) == find(b)` |
| Word Ladder | BFS su parole, vicini = una lettera diversa | pattern con jolly `h*t`, `defaultdict(list)` |

## Sintassi al volo

| Serve | Sintassi |
|---|---|
| dimensioni griglia | `rows, cols = len(grid), len(grid[0])` |
| 4 direzioni | `DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]` |
| dentro la griglia? | `0 <= r < rows and 0 <= c < cols` |
| lista di adiacenza | `graph = defaultdict(list)` · `graph[a].append(b)` (+ `graph[b].append(a)` se non diretto) |
| visitati | `visited = set()` · `visited.add((r, c))` |
| coda BFS | `queue = deque([start])` · `queue.popleft()` |
| matrice di valori | `[[0] * cols for _ in range(rows)]` |

## Griglia: DFS con direzioni

Il pattern di "isole": per ogni cella non visitata che soddisfa la condizione, lancia una DFS che marca tutta la componente. Il numero di lanci è il numero di componenti.

In [1]:
grid = [
    "11000",
    "11010",
    "00011",
]
rows, cols = len(grid), len(grid[0])
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]
visited: set[tuple[int, int]] = set()

def dfs(r: int, c: int) -> int:
    if not (0 <= r < rows and 0 <= c < cols) or grid[r][c] != "1" or (r, c) in visited:
        return 0
    visited.add((r, c))
    return 1 + sum(dfs(r + dr, c + dc) for dr, dc in DIRS)

sizes = [dfs(r, c) for r in range(rows) for c in range(cols) if grid[r][c] == "1" and (r, c) not in visited]
print(sizes, len(sizes))  # -> [4, 3] 2

[4, 3] 2


Se la griglia è una lista di liste modificabile, puoi marcare in-place (`grid[r][c] = "0"`) invece di usare `visited`: meno memoria, ma modifichi l'input.

## BFS multi-sorgente per livelli

Metti nella coda **tutte** le sorgenti all'inizio: la BFS si espande da tutte contemporaneamente. Ogni giro del `while` è un livello (un minuto, un passo di distanza).

In [2]:
from collections import deque

# 2 = magazzino, 1 = negozio da raggiungere, 0 = vuoto
grid = [
    [2, 1, 1],
    [0, 1, 0],
    [1, 1, 2],
]
rows, cols = len(grid), len(grid[0])
queue = deque((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 2)
to_reach = sum(row.count(1) for row in grid)

steps = 0
while queue and to_reach:
    for _ in range(len(queue)):  # solo il livello corrente
        r, c = queue.popleft()
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                grid[nr][nc] = 2  # marca QUANDO metti in coda, non quando estrai
                to_reach -= 1
                queue.append((nr, nc))
    steps += 1
print(steps, to_reach)  # -> 2 0
# se alla fine to_reach > 0, qualche cella è irraggiungibile (Rotting Oranges: -1)

2 0


## Liste di archi → lista di adiacenza

In [3]:
from collections import defaultdict

edges = [(0, 1), (1, 2), (3, 4)]
graph: defaultdict[int, list[int]] = defaultdict(list)
for a, b in edges:
    graph[a].append(b)
    graph[b].append(a)  # non diretto
print(dict(graph))  # -> {0: [1], 1: [0, 2], 2: [1], 3: [4], 4: [3]}

# con n nodi numerati 0..n-1 va bene anche una lista di liste
n = 5
adj = [[] for _ in range(n)]
for a, b in edges:
    adj[a].append(b)
print(adj[1])  # -> [2]

{0: [1], 1: [0, 2], 2: [1], 3: [4], 4: [3]}
[2]


## Clonare: dict originale → copia

La mappa fa anche da `visited`: se un nodo è già nella mappa, restituisci la sua copia invece di crearne un'altra (i cicli non causano ricorsione infinita).

In [4]:
class Node:
    def __init__(self, val: int, neighbors: list["Node"] | None = None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []  # mai [] come default

a, b = Node(1), Node(2)
a.neighbors.append(b)
b.neighbors.append(a)  # ciclo

copies: dict[Node, Node] = {}
def clone(node: Node) -> Node:
    if node in copies:
        return copies[node]
    copy = Node(node.val)
    copies[node] = copy  # registra PRIMA di visitare i vicini
    copy.neighbors = [clone(nb) for nb in node.neighbors]
    return copy

c = clone(a)
print(c.val, c.neighbors[0].val, c.neighbors[0].neighbors[0] is c, c is a)  # -> 1 2 True False

1 2 True False


## Grafo diretto: ciclo e ordinamento topologico (Kahn)

Con dipendenze "per fare B serve A" (arco `A → B`): conta i **gradi entranti**, metti in coda i nodi a grado 0, e quando ne estrai uno decrementa i suoi successori. Se alla fine non hai elaborato tutti i nodi, c'è un ciclo.

⚠️ Leggi bene il verso: in Course Schedule `[a, b]` significa "per seguire `a` devi aver fatto `b`", quindi l'arco è `b → a`.

In [5]:
from collections import deque

n = 4
prereqs = [(1, 0), (2, 1), (3, 1)]  # (corso, prerequisito)
graph = [[] for _ in range(n)]
indegree = [0] * n
for course, pre in prereqs:
    graph[pre].append(course)
    indegree[course] += 1

queue = deque(i for i in range(n) if indegree[i] == 0)
order = []
while queue:
    node = queue.popleft()
    order.append(node)
    for nxt in graph[node]:
        indegree[nxt] -= 1
        if indegree[nxt] == 0:
            queue.append(nxt)
print(order, len(order) == n)  # -> [0, 1, 2, 3] True: nessun ciclo

[0, 1, 2, 3] True


**Alternativa DFS a 3 stati:** `0` = non visitato, `1` = nello stack di ricorsione, `2` = completato. Trovare un vicino in stato `1` significa ciclo. Il solo `visited` non basta nei grafi diretti: un nodo già completato non indica un ciclo.

## Union-Find

Tiene insiemi disgiunti: `find(x)` restituisce il rappresentante, `union(a, b)` unisce due insiemi e restituisce `False` se erano già uniti (= l'arco chiude un ciclo). Con **path compression** e **union by size** le operazioni sono quasi `O(1)`.

In [6]:
parent = list(range(5))
size = [1] * 5

def find(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]  # path compression (dimezzamento)
        x = parent[x]
    return x

def union(a: int, b: int) -> bool:
    ra, rb = find(a), find(b)
    if ra == rb:
        return False  # già connessi: questo arco crea un ciclo
    if size[ra] < size[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    size[ra] += size[rb]
    return True

results = [union(a, b) for a, b in [(0, 1), (1, 2), (3, 4), (2, 0)]]
print(results)                              # -> [True, True, True, False]
print(len({find(x) for x in range(5)}))     # -> 2 componenti

[True, True, True, False]
2


## Vicini "impliciti" con pattern

Quando i vicini non sono dati ma definiti da una regola ("differisce di una lettera"), precalcola dei **bucket**: ogni parola finisce sotto i pattern ottenuti sostituendo una lettera con `*`. Due parole nello stesso bucket sono vicine.

In [7]:
from collections import defaultdict

words = ["hot", "dot", "dog", "lot"]
buckets: defaultdict[str, list[str]] = defaultdict(list)
for word in words:
    for i in range(len(word)):
        buckets[word[:i] + "*" + word[i + 1 :]].append(word)
print(buckets["*ot"], buckets["do*"])  # -> ['hot', 'dot', 'lot'] ['dot', 'dog']

['hot', 'dot', 'lot'] ['dot', 'dog']


## Errori comuni

- Marcare come visitato quando **estrai** dalla coda invece che quando **inserisci**: lo stesso nodo entra più volte.
- `grid[r][c]` prima del controllo sui bordi: `IndexError` (o, con indici negativi, accesso silenzioso all'ultima riga!).
- Default mutabile `def f(x=[])`: la lista è condivisa tra chiamate.
- Nodi isolati: con `defaultdict` un nodo senza archi non compare tra le chiavi; itera su `range(n)`.
- Ricorsione profonda su griglie grandi (es. 300×300): preferisci la BFS iterativa.